In [5]:
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
DATA_PATH = Path("data/movie_dataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found at data/movie_dataset.csv. Please place the file there before running this notebook."
    )

movies = pd.read_csv(DATA_PATH, low_memory=False)
movies = movies.dropna(subset=["title"]).drop_duplicates(subset=["title"]).reset_index(drop=True)
print("Dataset shape:", movies.shape)

TEXT_FEATURE_COLUMNS = ["genres", "keywords", "tagline", "cast", "director", "overview"]
available_columns = [col for col in TEXT_FEATURE_COLUMNS if col in movies.columns]

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if not available_columns:
    raise ValueError("No expected text columns found in the dataset.")

text_data = movies[available_columns].fillna("").copy()
for col in available_columns:
    text_data[col] = text_data[col].map(clean_text)

corpus = text_data.agg(" ".join, axis=1)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
tfidf_matrix = vectorizer.fit_transform(corpus)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

Dataset shape: (4800, 24)
TF-IDF matrix shape: (4800, 31797)


In [3]:
def recommend_movies(movie_title: str, top_n: int = 10) -> tuple[str, pd.DataFrame]:
    exact_match = movies[movies["title"].str.lower() == movie_title.lower()]

    if exact_match.empty:
        partial = movies[movies["title"].str.contains(movie_title, case=False, regex=False)]
        if partial.empty:
            raise ValueError(f"Movie not found: {movie_title}")
        selected_index = int(partial.index[0])
        selected_title = str(partial.iloc[0]["title"])
    else:
        selected_index = int(exact_match.index[0])
        selected_title = str(exact_match.iloc[0]["title"])

    similarity_scores = linear_kernel(tfidf_matrix[selected_index], tfidf_matrix).flatten()
    ranked_indices = similarity_scores.argsort()[::-1]

    results = []
    for idx in ranked_indices:
        if idx == selected_index:
            continue

        results.append({
            "title": str(movies.iloc[idx]["title"]),
            "similarity": float(similarity_scores[idx])
        })

        if len(results) >= top_n:
            break

    return selected_title, pd.DataFrame(results)

In [4]:
selected_title, recommendations = recommend_movies("Avatar", top_n=10)
print(f"Top {len(recommendations)} recommendations for '{selected_title}':")
recommendations

Top 10 recommendations for 'Avatar':


,title,similarity
0,Aliens,0.137779
1,Cargo,0.121514
2,Star Trek Beyond,0.118196
3,Guardians of the Galaxy,0.116557
4,Alien,0.116499
5,Lockout,0.115161
6,Moonraker,0.113535
7,Gattaca,0.106150
8,Lifeforce,0.102491
9,Gravity,0.100790


In [7]:
# Optional interactive execution
movie_name = input("Enter a movie title: ")
top_n = int(input("Enter number of recommendations: "))
selected_title, recs = recommend_movies(movie_name, top_n=top_n)
print(f"Top {len(recs)} recommendations for '{selected_title}':")
recs

Top 5 recommendations for 'Avengers: Age of Ultron':


,title,similarity
0,The Avengers,0.493439
1,Captain America: Civil War,0.289600
2,Iron Man 2,0.265279
3,Iron Man,0.196827
4,Iron Man 3,0.179531
